In [2]:
import ee
import geemap

# Authenticate once if needed, then initialize Earth Engine.
# ee.Authenticate()
ee.Initialize()

# ---------------------------------------------------------
# 1. STUDY AREA (CENTRAL KALIMANTAN)
# ---------------------------------------------------------

gaul = ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level1")

roi = (
    gaul.filter(ee.Filter.eq("ADM0_NAME", "Indonesia"))
    .filter(ee.Filter.eq("ADM1_NAME", "Kalimantan Tengah"))
)

# Interactive map (works in a Jupyter notebook or similar environment).
Map = geemap.Map()
Map.centerObject(roi, 7)
Map.addLayer(roi, {}, "Central Kalimantan")

# ---------------------------------------------------------
# 2. REFERENCE ABOVEGROUND CARBON STOCK (2010)
# ---------------------------------------------------------

# NASA ORNL dataset is carbon density, not biomass.
agc = (
    ee.ImageCollection("NASA/ORNL/biomass_carbon_density/v1")
    .select("agb")  # Aboveground carbon density (Mg C/ha), per the source script.
    .first()
    .clip(roi)
)

agc_vis = {
    "min": 0,
    "max": 200,
    "palette": ["a7ffba", "33d066", "228b44", "0f3f1f", "000000"],
}
Map.addLayer(agc, agc_vis, "AGC Carbon Reference (NASA 2010)")

# ---------------------------------------------------------
# 3. BUILD PREDICTOR STACK
# ---------------------------------------------------------

def build_predictors(start_date, end_date, roi):
    """Build and return the clipped MODIS/topography predictor image."""

    # Vegetation indices
    def scale_vi(img):
        return img.multiply(0.0001).copyProperties(img, img.propertyNames())

    vi = (
        ee.ImageCollection("MODIS/061/MOD13Q1")
        .select(["NDVI", "EVI"])
        .filterDate(start_date, end_date)
        .map(scale_vi)
        .median()
    )

    # LAI and FPAR
    def scale_lai_fpar(img):
        lai = img.select("Lai_500m").multiply(0.1)
        fpar = img.select("Fpar_500m").multiply(0.01)
        return lai.addBands(fpar)

    lai_fpar = (
        ee.ImageCollection("MODIS/061/MOD15A2H")
        .select(["Lai_500m", "Fpar_500m"])
        .filterDate(start_date, end_date)
        .map(scale_lai_fpar)
        .median()
    )

    # Land cover
    lc = (
        ee.ImageCollection("MODIS/061/MCD12Q1")
        .select("LC_Type1")
        .filterDate(start_date, end_date)
        .mode()
    )

    # Elevation
    elevation = ee.Image("USGS/GTOPO30").select("elevation")

    return ee.Image.cat([vi, lai_fpar, lc, elevation]).clip(roi)


# Example:
# predictors = build_predictors("2010-01-01", "2011-01-01", roi)
# Map.addLayer(predictors.select("NDVI"), {"min": 0, "max": 1}, "NDVI")
# Map

ModuleNotFoundError: No module named 'ee'